# PROCESO DE EXTRACCIÓN, TRANSFORMACIÓN Y CARGA DE DATOS SOBRE MUNICIPIOS Y PROVINCIAS ESPAÑOLAS (ETL)

## 0. Carga de librerías esenciales para el proceso

In [566]:
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import json

## 1. Extracción de datos a través de APIs oficiales y públicas

En primer lugar, ejecutamos el main.py de extracción de datos, para actualizar mantener los datos actualizados.

Nota: El alumno recomienda no actualizarlo si ya se poseen los datos, ya que puede suponer un tiempo de espera elevado

In [567]:
actualizar="n"
#actualizar = input("""¿Deseas actualizar los archivos de datos? (s/n)
 #        Ten en cuenta que puede tardar bastante en hacer las consultas, 
  #       por lo que si ya has cargado los datos una vez recomiendo no volver a hacerlo: """)

if actualizar=="s":
    %run Extractor/main.py
elif actualizar=="n":
    print("No se han actualizado los archivos de datos.")

No se han actualizado los archivos de datos.


En la carga inicial de datos, es importante recalcar que municipiosDF y provinciasDF son extraidos manualmente del Instituto Geográfico Nacional, que no posee una API abierta al público. Se comentará más en el apartado 2.1 de este Notebook

In [568]:
# Información de flujos de movimiento
INE_localidades = pd.read_csv('Extractor/data/processed/flujo_ine_localidad.csv', sep=',', encoding='UTF-8')
f_INE_provincias = pd.read_csv('Extractor/data/processed/flujo_ine_provincia.csv', sep=',', encoding='UTF-8')
INE_provincias = pd.read_csv('Extractor/data/processed/INE_provincias.csv', sep=',', encoding='UTF-8')
municipios_data = pd.read_csv('Extractor/data/processed/municipios_espana.csv', sep=',', encoding='UTF-8')

# Información económica hotelera
rentabilidad_H = pd.read_csv('Extractor/data/processed/rentabilidad_hotelera.csv', sep=",", encoding= 'UTF_8')

#Información demográfica y geográfica de provincias y municipios

municipiosDF = pd.read_csv('Extractor/data/raw/MUNICIPIOS.csv', sep=';', encoding='latin1')
provinciasDF = pd.read_csv('Extractor/data/raw/PROVINCIAS.csv', sep=';', encoding='latin1')

## 2. Transformación de la información extraida

### 2.1 Datos geográficos de municipios y provincias

In [569]:
# Primero revisamos la información y nombres de columnas en nuestros archivos descargados directamente del CNIG
provinciasDF.info()
print('-------------------------------------------------------')
municipiosDF.info()

municipiosDF.head(1).T

<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 5 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   COD_PROV            52 non-null     int64
 1   PROVINCIA           52 non-null     str  
 2   COD_CA              52 non-null     int64
 3   COMUNIDAD_AUTONOMA  52 non-null     str  
 4   CAPITAL             52 non-null     str  
dtypes: int64(2), str(3)
memory usage: 2.2 KB
-------------------------------------------------------
<class 'pandas.DataFrame'>
RangeIndex: 8132 entries, 0 to 8131
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   COD_INE                   8132 non-null   int64
 1   ID_REL                    8132 non-null   int64
 2   COD_GEO                   8132 non-null   int64
 3   COD_PROV                  8132 non-null   int64
 4   PROVINCIA                 8132 non-null   str  
 5   NOMBRE

,0
COD_INE,1001000000
ID_REL,1010014
COD_GEO,1010
COD_PROV,1
PROVINCIA,Araba/Álava
NOMBRE_ACTUAL,Alegría-Dulantzi
POBLACION_MUNI,2961
SUPERFICIE,"1994,5872"
PERIMETRO,35069
COD_INE_CAPITAL,1001000101


Como la información de estos dos Dataframes está de por si bastante bien estructurada, no necesita ningún trabajo de transformación, por lo que procedemos a incluir en municipiosDF la información de su provincia para en futuros apartados crear ratios con esta información

In [570]:
#En primer lugar, vamos a combinar los dos DFs extraidos manualmente del centro de descargas del IGN 
# https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion

# Merge para unir información de provincia por su codigo
municipiosDF = municipiosDF.merge(provinciasDF[['COD_PROV','COD_CA', 'COMUNIDAD_AUTONOMA']], on='COD_PROV')

municipiosDF = municipiosDF.drop(['ID_REL', 'HOJA_MTN25', 'ORIGENCOOR', 'ORIGENALTITUD'], axis=1)

municipiosDF.head()


,COD_INE,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ALTITUD,COD_CA,COMUNIDAD_AUTONOMA
0,1001000000,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,"-2,512507724","42,84045247",568,16,País Vasco/Euskadi
1,1002000000,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,"-3,001015194","43,05265767",217,16,País Vasco/Euskadi
2,1003000000,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,"-2,564829379","43,05257873",325,16,País Vasco/Euskadi
3,1004000000,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,"-3,13052099","43,1217919",199,16,País Vasco/Euskadi
4,1006000000,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,"-2,872270813","42,72340924",466,16,País Vasco/Euskadi


### 2.2 Datos de caracter turístico extraidos mediante la API del Instituto Nacional de Estadística (INE)

Comenzamos con una revisión general de los datos. 
Como la API nos devuelve una estructura similar para todos estos Dataframes, el proceso va a ser similar, pero con casos especiales para cada situación.

In [571]:
#INE_localidades.head()
#INE_localidades.shape --> Resultado: (6610, 11)
INE_localidades.info()

<class 'pandas.DataFrame'>
RangeIndex: 6610 entries, 0 to 6609
Data columns (total 12 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   COD                     6610 non-null   str    
 1   Nombre                  6610 non-null   str    
 2   T3_Unidad               6610 non-null   str    
 3   T3_Escala               6610 non-null   str    
 4   Fecha                   6610 non-null   str    
 5   T3_Periodo              6610 non-null   str    
 6   T3_TipoDato             6610 non-null   str    
 7   Anyo                    6610 non-null   int64  
 8   Valor                   5560 non-null   float64
 9   MetaData_json           6610 non-null   str    
 10  tabla_id                6610 non-null   int64  
 11  _meta.fecha_extraccion  6610 non-null   str    
dtypes: float64(1), int64(2), str(9)
memory usage: 619.8 KB


In [572]:
INE_provincias.head()
#INE_provincias[INE_provincias['INE_EOH_PROV.indicador']=='Viajero'].head()
#INE_provincias.info()

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,DPOP1,Total Nacional. Total. Total habitantes. Perso...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,47385107.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
1,DPOP2,Total Nacional. Hombres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,23222953.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
2,DPOP3,Total Nacional. Mujeres. Total habitantes. Per...,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,24162154.0,"[{""Id"": 16473, ""T3_Variable"": ""Comunidades y C...",2852,2026-09-07T22:13:34.997660
3,DPOP160,Albacete. Total. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,386464.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-07T22:13:34.997660
4,DPOP161,Albacete. Hombres. Total habitantes. Personas.,Personas,,2021-01-01T00:00:00.000+01:00,A,Definitivo,2021,193205.0,"[{""Id"": 3, ""T3_Variable"": ""Provincias"", ""Nombr...",2852,2026-09-07T22:13:34.997660


Cambios necesarios:
- INE_localidades y localidades_data se deben combinar
- En f_INE_provincias, INE_Provincias, Provincias_data y OpenStreetDF se deben combinar

El resultado debe ser dos DataFrames, uno para localidades, y otro para provincias. Posteriormente se deberá hacer un Left Join en las localidades para los municipios. Posteriormente, se deben combinar ambos Dfs

Comenzamos filtrando y reduciendo las columnas en localidades_data

In [573]:
municipios_data.nunique()

COD                       24396
Nombre                    24345
T3_Unidad                     1
T3_Escala                     1
Fecha                         1
T3_Periodo                    1
T3_TipoDato                   1
Anyo                          1
Valor                      6179
MetaData_json             24396
tabla_id                      1
_meta.fecha_extraccion        1
dtype: int64

In [574]:
#Revisamos el interior de los json que encontraremos en gran parte de los datasets trabajados
pd.set_option('display.max_colwidth', None) # Comando para poder ver cadenas largas de texto
print(municipios_data['MetaData_json'].tail())
      
pd.reset_option('display.max_colwidth') # Retrocedemos el comando

24391     [{"Id": 6270, "T3_Variable": "Municipios", "Nombre": "Zúñiga", "Codigo": "31265"}, {"Id": 452, "T3_Variable": "Sexo", "Nombre": "Hombres", "Codigo": "1"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo de dato", "Nombre": "Personas", "Codigo": ""}]
24392     [{"Id": 6270, "T3_Variable": "Municipios", "Nombre": "Zúñiga", "Codigo": "31265"}, {"Id": 453, "T3_Variable": "Sexo", "Nombre": "Mujeres", "Codigo": "2"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo de dato", "Nombre": "Personas", "Codigo": ""}]
24393      [{"Id": 5824, "T3_Variable": "Municipios", "Nombre": "Zurgena", "Codigo": "04103"}, {"Id": 451, "T3_Variable": "Sexo", "Nombre": "Total", "Codigo": "0"}, {"Id": 8677, "T3_Variable": "Tamaño de los municipios", "Nombre": "Total habitantes", "Codigo": "0"}, {"Id": 20258, "T3_Variable": "Tipo 

Observamos que el Código identificador de INE se encuentra en una columna de multiples Json en la columna MetaData_json. Lo mejor será definir una funcion para extraer esta información adecuadamente

In [575]:
def obtener_codigo_INE(df):
    """Formula definida para extraer un valor específico de una columna específica de varios dfs
    El valor extraido es el ultimo del primer diccionario que encontramos"""
    try:
        elementos = json.loads(df) if isinstance(df, str) else df
        if isinstance(elementos, list):
            for item in elementos:
                variable = item.get('T3_Variable', '').lower()
                # Sirve tanto para 'PUNTOS TURÍSTICOS' (flujo) como para 'Municipios' (padrón)
                if 'punto' in variable or 'muni' in variable:
                    return item.get('Codigo')
    except (json.JSONDecodeError, TypeError, IndexError):
        return None
    return None

def obtener_codigo_provincia_INE(df):
    try:
        elementos = json.loads(df) if isinstance(df, str) else df
        if isinstance(elementos, list):
            for item in elementos:
                # Buscamos únicamente la dimensión de Provincias
                if 'prov' in item.get('T3_Variable', '').lower():
                    return item.get('Codigo')
    except Exception:
        return None
    return None

In [576]:
# Primero, dividimos la columna de 'Nombre' usando el metodo .str.split()
columnas = ['Localidad', 'Genero', 'Metrica', 'Unidad']
municipios_data[columnas] = municipios_data['Nombre'].str.split('.', n=3, expand=True)

for col in columnas:
    municipios_data[col] = municipios_data[col].str.strip() #Eliminamos los espacios al inicio y final con un bucle

# A continuación, extraemos el codigo INE del json en la tabla
municipios_data['COD_INE'] = municipios_data['MetaData_json'].apply(obtener_codigo_INE)

# Suprimimos las columnas que no vamos a necesitar
municipios_data.drop(columns=['COD', 'Nombre', 'Fecha', 'T3_Unidad', 'T3_Escala', 'T3_Periodo', 'T3_TipoDato','Anyo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                     )

# Como nos interesa la poblacion total por municipio, filtramos Genero == 'Total
municipios_data = municipios_data[municipios_data['Genero'] == 'Total']

# Comprobamos el resultado
print(municipios_data.head())

print('--------------------------------------------------------------------------------------------------------')
print('Nota: realmente solo nos interesa COD_INE, Valor y localidad, pero el resto de columnas aportan contexto')

     Valor Localidad Genero           Metrica     Unidad COD_INE
0     73.0    Ababuj  Total  Total habitantes  Personas.   44001
3    849.0    Abades  Total  Total habitantes  Personas.   40001
6    337.0    Abadía  Total  Total habitantes  Personas.   10001
9   2239.0    Abadín  Total  Total habitantes  Personas.   27001
12  7768.0   Abadiño  Total  Total habitantes  Personas.   48001
--------------------------------------------------------------------------------------------------------
Nota: realmente solo nos interesa COD_INE, Valor y localidad, pero el resto de columnas aportan contexto


In [577]:
# Comprobamos si hemos extraido bien COD_INE
print( """COMPROBACIÓN DE municipios_data
      """)
print(f"Valores nulos: {municipios_data['COD_INE'].isna().sum()}")
print('--------------------------------')
print(f"Valores duplicados (esperados {municipios_data['COD_INE'].count()}): {municipios_data['COD_INE'].nunique()}")

COMPROBACIÓN DE municipios_data
      
Valores nulos: 0
--------------------------------
Valores duplicados (esperados 8132): 8132


A continuación, adaptaremos INE_localidades.

En la columna T3_Unidad tenemos dos valores, Viajeros y pernoctaciones, por lo que nos interesa quedarnos con ambos pero en distintas columnas.

In [578]:
INE_localidades.head(2) #2 para que ocupe poco espacio en la salida

,COD,Nombre,T3_Unidad,T3_Escala,Fecha,T3_Periodo,T3_TipoDato,Anyo,Valor,MetaData_json,tabla_id,_meta.fecha_extraccion
0,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-07-01T00:00:00.000+02:00,M07,Provisional,2026,18734.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-07T22:13:49.539679
1,EOT2611,Nacional. Viajeros. Vitoria-Gastéiz. Residente...,Viajeros,,2026-06-01T00:00:00.000+02:00,M06,Provisional,2026,19464.0,"[{""Id"": 284332, ""T3_Variable"": ""Concepto turís...",2078,2026-09-07T22:13:49.539679


In [579]:
#Como el DF tiene dos niveles en el mismo origen de datos, vamos a partirlo en dos y despues unir sus columnas resultantes.

#El primer nivel será en INE_localidades2
INE_localidades2 = INE_localidades[(~INE_localidades['Nombre'].str.startswith('Nacional'))]
columnas = ['localidad', 'metrica', 'Campos', 'Origen', 'Tipo']

# Split de las columnas iniciales
INE_localidades2[columnas] = INE_localidades2['Nombre'].str.split('.', n=4, expand=True)

# Extraemos COD_INE del json
INE_localidades2['COD_INE'] = INE_localidades2['MetaData_json'].apply(obtener_codigo_INE)

# Borramos columnas que no necesitamos (aunque despues pivotaremos, por lo que no es estrictamente necesario, mejora los siguientes pasos)
INE_localidades2.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion', 'Tipo'],
                     inplace=True
                     )

# Exploración de distintas métricas
print('Valores únicos en cada columna:')
print(INE_localidades2.nunique()) 
print('-----------------------------')
print(INE_localidades2.Anyo.value_counts())
print('-----------------------------')
print(INE_localidades2.metrica.value_counts())
print('-----------------------------')
print(INE_localidades2.Origen.value_counts())
print('-----------------------------')
# Conclusiones:
 # Solo nos interesa mantener viajeros y pernoctaciones (Total categorias es el total)
 # Nos interesa diferenciar el origen en dos columnas, pero cambiando los nombres


INE_localidades2 = (INE_localidades2[(INE_localidades2['metrica'] != ' Total categorías')]
                    .rename(columns= {'T3_Periodo': 'periodo'}) # Renombramos la columna para su posterior uso
                    )

INE_localidades2['Origen'] = (INE_localidades2['Origen']
                              .str.strip() #Para limpiar el texto, ya que .replace solo no daba resultado
                              .replace({'Residentes en España': 'Nacional',
                                        'Residentes en el Extranjero': 'Extranjero'}))


INE_localidades2 =(INE_localidades2.sort_values('Valor', ascending=False)
 .pivot_table(index= ['COD_INE', 'localidad'],
                             columns= ['metrica', 'Origen'],
                             values= ['Valor', 'periodo'],
                             aggfunc= {'Valor': 'sum', 'periodo': 'first'})
 .reset_index())

INE_localidades2.columns = INE_localidades2.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('Resultado del Dataframe')
INE_localidades2.head(3)


Valores únicos en cada columna:
COD            164
T3_Unidad        2
T3_Periodo      12
Anyo             2
Valor         1350
localidad       40
metrica          3
Campos           4
Origen           2
COD_INE         41
dtype: int64
-----------------------------
Anyo
2026    1142
2025     812
Name: count, dtype: int64
-----------------------------
metrica
Viajero             905
Pernoctaciones      905
Total categorías    144
Name: count, dtype: int64
-----------------------------
Origen
Residentes en España           978
Residentes en el Extranjero    976
Name: count, dtype: int64
-----------------------------
Resultado del Dataframe


,COD_INE,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,03018,Altea,178244.0,149324.0,59581.0,72404.0,M09,M08,M05,M07
1,04032,Carboneras,5008.0,52569.0,2055.0,21598.0,M09,M08,M09,M08
2,04902,"Ejido, El",0.0,0.0,0.0,0.0,M07,M07,M07,M07


In [580]:
#El segundo nivel mencionado anteriormente será en INE_localidades_nac, con los valores de residentes en españa y extranjeros
# Ahora, repetimos con la otra parte del dataframe
INE_localidades_nac = INE_localidades[INE_localidades['Nombre'].str.startswith('Nacional')]

# Misma operación con la otra metrica, pero diferencia en las columnas que desagregamos
columnas= ['tipo', 'metrica', 'localidad', 'tipo residente']

INE_localidades_nac[columnas] = INE_localidades_nac['Nombre'].str.split('.', n=3, expand=True)

# Funcion de MetaData_json
INE_localidades_nac['COD_INE'] = INE_localidades_nac['MetaData_json'].apply(obtener_codigo_INE)

INE_localidades_nac.drop(columns=['Nombre', 'Fecha', 'T3_Escala', 'T3_TipoDato','tipo', 'tabla_id', 'MetaData_json', '_meta.fecha_extraccion'],
                     inplace=True
                        )

# Revisamos un poco los valores que nos encontramos
print(INE_localidades_nac.nunique()) 
print('-----------------------------')
print(INE_localidades_nac.metrica.value_counts()) #Otra vez, elegimos 2026
print('-----------------------------')
print(INE_localidades_nac['tipo residente'].value_counts()) #Genera un problema, ya que algunas filas no presentan bien si es español o extranjero

#Primero, solucionaremos el problema de tipo residente, apoyandonos en numpy

condiciones = [INE_localidades_nac['tipo residente'].str.contains('España', case=False, na = False),
               INE_localidades_nac['tipo residente'].str.contains('extranjero', case=False, na = False)
               ]
eleccion = ['Nacional', 'Extranjero']

INE_localidades_nac['tipo turista'] = np.select(condiciones, eleccion, default= ' ')

# Modificamos el DF
INE_localidades_nac = (INE_localidades_nac[(INE_localidades_nac['Anyo'] == 2026) & #Nos quedamos con 2026 porque tiene más metricas
                                           (INE_localidades_nac['metrica'] != ' Establecimientos hoteleros')
                                           ] 
                       .rename(columns={'T3_Periodo': 'periodo'}) #Aprovechamos para renombrar esta columna
                       )                 

# metodo rapido para modificar valor, ya que será necesario en el siguiente
INE_localidades_nac.loc[INE_localidades_nac['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'

# Para lograr tener el mes de mayor valor en la pivotacion, primero ordenaremos para poder mantener el mes con mayor numero de viajeros
INE_localidades_nac = (INE_localidades_nac
                       .sort_values('Valor', ascending=False) #Ordenamos
                       .pivot_table(index=['COD_INE','localidad'],
                                    columns= ['metrica', 'tipo turista'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'}) #Pivotamos
                       .reset_index()
                       )

# Eliminamos bandas y unificamos en el nombre de columna
INE_localidades_nac.columns = INE_localidades_nac.columns.map(' '.join) #join con valor vacio porque ya existe un valor vacio en las metricas

print('-----------------------------')
print('Resultado del Dataframe:')
INE_localidades_nac.head(3)


COD                388
T3_Unidad            2
T3_Periodo          12
Anyo                 2
Valor             4077
metrica              3
localidad           74
tipo residente      52
COD_INE             97
dtype: int64
-----------------------------
metrica
Viajeros                      1728
Pernoctaciones                1728
Establecimientos hoteleros    1200
Name: count, dtype: int64
-----------------------------
tipo residente
Residentes en España.                                             1728
Residentes en el extranjero.                                      1728
03063-Denia. Residentes en España.                                  24
03063-Denia. Residentes en el extranjero.                           24
04066-Níjar. Residentes en España.                                  24
04066-Níjar. Residentes en el extranjero.                           24
07014-Capdepera. Residentes en España.                              24
07014-Capdepera. Residentes en el extranjero.                       2

,COD_INE,localidad,Valor Pernoctaciones Extranjero,Valor Pernoctaciones Nacional,Valor Viajero Extranjero,Valor Viajero Nacional,periodo Pernoctaciones Extranjero,periodo Pernoctaciones Nacional,periodo Viajero Extranjero,periodo Viajero Nacional
0,01059,Vitoria-Gastéiz,175065.0,293398.0,57422.0,136922.0,M07,M04,M07,M04
1,03014,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06
2,03031,Benidorm,4105484.0,2238270.0,834735.0,551668.0,M07,M07,M05,M07


Antes de concatenar (anexar) de vuelta los dos dataframes, debemos solucionar la diferencia entre las columnas resultado las operaciones paralelas.

Hay que tener en cuenta que algunos códigos INE son un valor alfanumérico origen de la encuesta EOH. debemos transformar este código al codigo INE de la provincia, o en su defecto realizar una combinación de columnas a partir del nombre

In [581]:
#Unificados nombres de columnas
columnas = (INE_localidades_nac.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )

# Aplicamos
INE_localidades2.columns = columnas
INE_localidades_nac.columns = columnas

# Unimos los Dfs
INE_localidades = pd.concat([INE_localidades2, INE_localidades_nac], ignore_index=True)

# Filtramos para suprimir que no tienen valores
INE_localidades = INE_localidades[INE_localidades[INE_localidades.columns[2]] > 0]


# Revisamos
INE_localidades.sort_values('LOCALIDAD')


,COD_INE,LOCALIDAD,VALOR PERNOCTACIONES EXTRANJERO,VALOR PERNOCTACIONES NACIONAL,VALOR VIAJERO EXTRANJERO,VALOR VIAJERO NACIONAL,PERIODO PERNOCTACIONES EXTRANJERO,PERIODO PERNOCTACIONES NACIONAL,PERIODO VIAJERO EXTRANJERO,PERIODO VIAJERO NACIONAL
47,A1,Adeje,6069924.0,302222.0,859460.0,76113.0,M07,M07,M03,M07
48,A5,Albacete,32020.0,171794.0,16601.0,111280.0,M02,M05,M04,M05
49,A6,Albarracín,6936.0,37844.0,4255.0,20366.0,M05,M04,M03,M04
50,A8,Algeciras,73041.0,88538.0,43367.0,43260.0,M07,M03,M03,M03
39,03014,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06
...,...,...,...,...,...,...,...,...,...,...
12,07061,Sóller,471032.0,20453.0,123481.0,8635.0,M08,M06,M06,M06
31,K7,Teguise,2182897.0,321562.0,294602.0,64153.0,M10,M08,M10,M08
24,25043,"Vall de Boí, La",4950.0,83094.0,2443.0,27839.0,M08,M08,M08,M08
32,M4,Yaiza,4508668.0,562652.0,612714.0,95684.0,M10,M08,M10,M08


Pasando a los datos agregados por provincias, el flujo de transformaciones será similar a los dfs de localidades, con pequeñas variaciones

In [582]:
#Revisamos el interior de los json que encontraremos en gran parte de los datasets trabajados
pd.set_option('display.max_colwidth', None) # Comando para poder ver cadenas largas de texto
print(f_INE_provincias['MetaData_json'].tail())
      
pd.reset_option('display.max_colwidth') # Retrocedemos el comando

5035    [{"Id": 8995, "T3_Variable": "Comunidades y Ciudades Autónomas", "Nombre": "Melilla", "Codigo": "19"}, {"Id": 284333, "T3_Variable": "Concepto turístico", "Nombre": "Pernoctaciones", "Codigo": "C"}, {"Id": 9834, "T3_Variable": "TIPO DE CATEGORIA", "Nombre": "Total categorías", "Codigo": ""}, {"Id": 19965, "T3_Variable": "RESIDENCIA/ORIGEN", "Nombre": "Residentes en el Extranjero", "Codigo": ""}, {"Id": 72, "T3_Variable": "Tipo de dato", "Nombre": "Dato", "Codigo": "0"}]
5036    [{"Id": 8995, "T3_Variable": "Comunidades y Ciudades Autónomas", "Nombre": "Melilla", "Codigo": "19"}, {"Id": 284333, "T3_Variable": "Concepto turístico", "Nombre": "Pernoctaciones", "Codigo": "C"}, {"Id": 9834, "T3_Variable": "TIPO DE CATEGORIA", "Nombre": "Total categorías", "Codigo": ""}, {"Id": 19965, "T3_Variable": "RESIDENCIA/ORIGEN", "Nombre": "Residentes en el Extranjero", "Codigo": ""}, {"Id": 72, "T3_Variable": "Tipo de dato", "Nombre": "Dato", "Codigo": "0"}]
5037    [{"Id": 8995, "T3_Variable

In [583]:
# Comenzamos revisando los
print(f_INE_provincias.nunique())
print('-----------------------------')
print(f_INE_provincias['T3_Unidad'].value_counts())
print('-----------------------------')
print(f_INE_provincias.Nombre.value_counts())

#NOTA: en este DF están mezcladas provincias, comunidades y total nacional, conviene quedarse solo con provincias
# Para quedarnos con Provincias, se debe filtrar en la columna MetaData_json, que contiene el valor crudo en formato json, aunque no es necesario desagregarlo

f_INE_provincias = f_INE_provincias[f_INE_provincias['MetaData_json'].str.contains('Provincia', case=False, na=False)]

columnas=['Provincia', 'metrica', 'Origen']

f_INE_provincias[columnas] = f_INE_provincias['Nombre'].str.split('.', n=2, expand=True)

# Extraemos COD_PROV del json, valor numerico que define la provincia
f_INE_provincias['COD_PROV'] = f_INE_provincias['MetaData_json'].apply(obtener_codigo_provincia_INE)

f_INE_provincias = f_INE_provincias.drop(columns= ['Nombre', 'T3_Escala', 'Fecha', 'T3_TipoDato', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion'])


#Filtramos años. En este caso nos quedamos con 2025 ya que contiene los doce meses
f_INE_provincias = (f_INE_provincias[f_INE_provincias['Anyo'] == 2025]
                     .rename(columns={'T3_Periodo': 'periodo'}))
                     
f_INE_provincias=f_INE_provincias.sort_values('Valor', ascending=False)


f_INE_provincias.loc[f_INE_provincias['metrica'].str.strip() == 'Viajeros', 'metrica'] = 'Viajero'
f_INE_provincias['metrica'] = f_INE_provincias['metrica'].str.strip()

f_INE_provincias = f_INE_provincias.pivot_table(index=['COD_PROV', 'Provincia'],
                                    columns= ['metrica'],
                                    values= ['Valor','periodo'],
                                    aggfunc= {'Valor':'sum', 'periodo':'first'})

f_INE_provincias.columns = f_INE_provincias.columns.map(' '.join)
f_INE_provincias.columns = (f_INE_provincias.columns
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace('  ', ' ', regex= False) # suprimimos dobles espaciados
            .str.upper() #Normalizamos todas las columnas a mayusculas
            )

f_INE_provincias = f_INE_provincias.reset_index()
f_INE_provincias.head()

COD                        420
Nombre                     420
T3_Unidad                    2
T3_Escala                    1
Fecha                       12
T3_Periodo                  12
T3_TipoDato                  1
Anyo                         2
Valor                     4504
MetaData_json              420
tabla_id                     1
_meta.fecha_extraccion       1
dtype: int64
-----------------------------
T3_Unidad
Viajeros          2520
Pernoctaciones    2520
Name: count, dtype: int64
-----------------------------
Nombre
Nacional. Viajeros. Total categorías. Total.                          12
Nacional. Viajeros. Total categorías. Residentes en España.           12
Nacional. Viajeros. Total categorías. Residentes en el extranjero.    12
Nacional. Pernoctaciones. Total categorías. Total.                    12
Nacional. Pernoctaciones. Total categorías. Residentes en España.     12
                                                                      ..
Melilla. Viajeros. Residente

,COD_PROV,Provincia,VALOR PERNOCTACIONES,VALOR VIAJERO,PERIODO PERNOCTACIONES,PERIODO VIAJERO
0,01,Alava,982555.0,456280.0,M08,M08
1,02,Albacete,802323.0,373917.0,M09,M09
2,03,Alicante,16590966.0,4345693.0,M08,M08
3,04,Almería,4567704.0,1372225.0,M08,M08
4,05,Avila,563816.0,355110.0,M08,M08


Para la rentabilidad hotelera, es importante conocer los dos indicadores que comparte este DataFrame
- **ADR (Average Daily Rate)**: Es el ingreso promedio por habitación ocupada (las habitaciones no alguiladas no cuentan)
- **RevPar (Revenue Per Available Room)**: Ingreso promedio por habitación disponible (ocupadas y disponibles).

In [584]:
#rentabilidad_H.info()

rentabilidad_H = rentabilidad_H.copy()

rentabilidad_H = rentabilidad_H[rentabilidad_H['MetaData_json'].str.contains('Provincias')]
columnas = ['Provincia', 'Metrica', 'Categoria', 'TipoDato']

rentabilidad_H[columnas] = rentabilidad_H['Nombre'].str.split('.', n=3, expand=True)

#Normalizamos nombres de columnas y posteriormente suprimimos las que no necesitamos
for col in columnas:
    rentabilidad_H[col] = (rentabilidad_H[col]
            .str.strip() #Eliminamos espacios al inicio y final
            .str.replace(r'\s+', ' ', regex= True) # suprimimos dobles espaciados
            .str.replace(r'[^\w\s]', '', regex= True)) # Truco para eliminar signos de puntuación

# Similar al anterior, usamos segunda formula para cod de provincia
rentabilidad_H['COD_INE'] = rentabilidad_H['MetaData_json'].apply(obtener_codigo_provincia_INE)

rentabilidad_H = rentabilidad_H.drop(['Nombre', 'T3_TipoDato', 'T3_Escala', 'T3_Unidad', 'MetaData_json', 'tabla_id', '_meta.fecha_extraccion', 'Categoria'], axis=1)


#Primero, revisamos los datos que nos podemos encontrar
print(rentabilidad_H.nunique())

# Renombramos para acortar
rentabilidad_H['Metrica'] = rentabilidad_H['Metrica'].replace({'Ingresos por habitación disponible RevPAR': 'RevPar',
                                            'Tarifa media diaria ADR':'ADR'})

# Reemplazamos para acortar TipoDato
rentabilidad_H['TipoDato'] =rentabilidad_H['TipoDato'].replace({'Tasa de variación interanual': 'Tasa Var'})

# Ordenamos  por valor mas algo y renombramos Periodo, para la posterior pivotacion
rentabilidad_H = (rentabilidad_H
         .sort_values('Valor', ascending= False)
         .rename(columns={'T3_Periodo':'Periodo'})
         )


# Sacamos la media de ADR por provincia como un dato extra.
# Es importante remarcar que ADR es ya de por si un ingreso medio de habitaciones ocupadas
# Por lo tanto, hacer la media por provincia no desvirtua tanto el dato como la media de RevPar 
rentAnual=(rentabilidad_H
           .groupby(['Provincia', 'TipoDato', 'Metrica'], as_index=False)['Valor']
           .mean()
           )

# Filtramos los datos para quedarnos con el ADR medio de cada provincia
rentAnual = rentAnual[(rentAnual['Metrica'] == 'ADR') & (rentAnual['TipoDato'] == 'Dato') ]

rentAnual = rentAnual.rename(columns={'Valor':'ADR Anual Medio'})
rentAnual = rentAnual.drop(['TipoDato', 'Metrica'], axis= 1)

# Pivotamos tablas, como hemos ordenado por valores mas alto podemos mantener Valor y periodo en first
rentabilidad_H = rentabilidad_H.pivot_table(index=['COD_INE', 'Provincia'],
                          columns=['Metrica','TipoDato'],
                          values=['Valor', 'Periodo'],
                          aggfunc={'Valor':'first', 'Periodo':'first'},
                          ) #Como ADR y RevPar son indicadores, no tiene sentido sumarlos

# Unimos las bandas y columnas para tener un unico nivel de columnas
rentabilidad_H.columns = rentabilidad_H.columns.map(' '.join)

# Reseteamos indices (Provincia)
rentabilidad_H = rentabilidad_H.reset_index()

rentabilidad_H.head()

COD            208
Fecha           12
T3_Periodo      12
Anyo             2
Valor         2182
Provincia       52
Metrica          2
TipoDato         2
COD_INE         52
dtype: int64


,COD_INE,Provincia,Periodo ADR Dato,Periodo ADR Tasa Var,Periodo RevPar Dato,Periodo RevPar Tasa Var,Valor ADR Dato,Valor ADR Tasa Var,Valor RevPar Dato,Valor RevPar Tasa Var
0,01,ArabaÁlava,M07,M06,M07,M10,117.12,13.91,89.31,27.88
1,02,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79
2,03,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54
3,04,Almería,M08,M09,M08,M09,163.14,16.26,137.15,19.80
4,05,Ávila,M04,M06,M08,M03,75.19,6.75,36.13,30.83


In [585]:
# Unimos las dos tablas trabjadas en la celda anterior
rentabilidad_H = rentabilidad_H.merge(rentAnual, how= 'left', on= 'Provincia', suffixes=('', '_A'))

# Borramos rentAnual para ahorrar memoria
del rentAnual

#rentabilidad_H = rentabilidad_H.drop(['Periodo RevPar Tasa Var' ], axis= 1 )
rentabilidad_H.info()

rentabilidad_H.head()


<class 'pandas.DataFrame'>
RangeIndex: 52 entries, 0 to 51
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   COD_INE                  52 non-null     str    
 1   Provincia                52 non-null     str    
 2   Periodo ADR Dato         52 non-null     str    
 3   Periodo ADR Tasa Var     52 non-null     str    
 4   Periodo RevPar Dato      52 non-null     str    
 5   Periodo RevPar Tasa Var  52 non-null     str    
 6   Valor ADR Dato           52 non-null     float64
 7   Valor ADR Tasa Var       52 non-null     float64
 8   Valor RevPar Dato        52 non-null     float64
 9   Valor RevPar Tasa Var    52 non-null     float64
 10  ADR Anual Medio          52 non-null     float64
dtypes: float64(5), str(6)
memory usage: 4.6 KB


,COD_INE,Provincia,Periodo ADR Dato,Periodo ADR Tasa Var,Periodo RevPar Dato,Periodo RevPar Tasa Var,Valor ADR Dato,Valor ADR Tasa Var,Valor RevPar Dato,Valor RevPar Tasa Var,ADR Anual Medio
0,01,ArabaÁlava,M07,M06,M07,M10,117.12,13.91,89.31,27.88,92.070833
1,02,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79,66.487500
2,03,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54,106.207500
3,04,Almería,M08,M09,M08,M09,163.14,16.26,137.15,19.80,89.287500
4,05,Ávila,M04,M06,M08,M03,75.19,6.75,36.13,30.83,69.874167


## Union de los distintos dataframes a dos niveles: provincia y municipio

El mayor reto al que nos enfrentamos es que, aunque tenemos diferentes identificadores para muchos de los dataframes, no se comparten entre todos, asi que tendremos que encontrar la forma de relacionarlos correctamente

Comenzamos repasando lo que tenemos: Los municipios funcionan con `COD_INE`, valor de 6 dígitos.

Las provincias funcionan con `COD_PROV`, con un valor de dos dígitos (del 01 al 50)

Tenemos además algunos valores por el código EOH, que no tiene una transformación oficial del código INE  a EOH, por lo que se debe aproximar por el nombre

In [586]:
# Observamos que seguimos con el problema de que algunos municipios aparecen con un código interno del INE, debemos corregirlo
INE_localidades[INE_localidades['COD_INE'].str.len() < 4]

,COD_INE,LOCALIDAD,VALOR PERNOCTACIONES EXTRANJERO,VALOR PERNOCTACIONES NACIONAL,VALOR VIAJERO EXTRANJERO,VALOR VIAJERO NACIONAL,PERIODO PERNOCTACIONES EXTRANJERO,PERIODO PERNOCTACIONES NACIONAL,PERIODO VIAJERO EXTRANJERO,PERIODO VIAJERO NACIONAL
28,B1,Almuñécar,333276.0,418600.0,52614.0,138623.0,M10,M08,M05,M08
29,B4,Antigua,2134591.0,155662.0,296738.0,47979.0,M08,M08,M03,M08
30,H4,"Oliva, La",2483771.0,229864.0,332412.0,52527.0,M08,M08,M10,M08
31,K7,Teguise,2182897.0,321562.0,294602.0,64153.0,M10,M08,M10,M08
32,M4,Yaiza,4508668.0,562652.0,612714.0,95684.0,M10,M08,M10,M08
...,...,...,...,...,...,...,...,...,...,...
105,L6,Valladolid,88514.0,320994.0,50952.0,190530.0,M07,M06,M07,M06
106,M1,Vigo,165676.0,311203.0,95080.0,156371.0,M07,M07,M05,M07
107,M5,Zamora,12288.0,90476.0,8395.0,55879.0,M05,M04,M05,M04
108,M6,Ávila,51951.0,197429.0,35636.0,129044.0,M05,M05,M05,M05


In [587]:

# Diccionario para pasar del código EOH a Cod INE
map_eoh_a_ine = {
    'A1': '38001', 'A5': '02003', 'A6': '44009', 'A8': '11004', 'B0': '04013',
    'B1': '18017', 'B4': '35003', 'B7': '38006', 'C2': '29025', 'C4': '48020',
    'C6': '09059', 'D1': '30016', 'D4': '15030', 'D6': '16078', 'D8': '10037',
    'D9': '11012', 'E0': '14021', 'E2': '20069', 'E3': '11027', 'E6': '29054',
    'E7': '46131', 'E8': '33024', 'E9': '18087', 'F4': '11020', 'F5': '35016',
    'F6': '24089', 'F7': '33036', 'F9': '25120', 'G0': '26089', 'G1': '27028',
    'G2': '28079', 'G3': '29069', 'G6': '35012', 'G7': '04064', 'H0': '30030',
    'H1': '29067', 'H2': '06083', 'H3': '29075', 'H4': '35014', 'H6': '33044',
    'H7': '35015', 'H9': '31201', 'I4': '38028', 'I6': '27051', 'I7': '29084',
    'I9': '37274', 'J0': '35019', 'J5': '38038', 'J6': '39075', 'J7': '15078',
    'K0': '40194', 'K2': '41091', 'K3': '42173', 'K4': '11035', 'K5': '43148',
    'K7': '35024', 'K8': '44216', 'L0': '35028', 'L1': '45168', 'L2': '29901',
    'L5': '46250', 'L6': '47186', 'M1': '36057', 'M4': '35034', 'M5': '49275',
    'M6': '05019', 'M7': '50297', 'M9': '11014', 'Q4': '13034', 'Q5': '17079',
    'Q7': '19257', 'R2': '37107'
}


# Reemplazar los códigos cortos por los códigos INE
INE_localidades['COD_INE'] = INE_localidades['COD_INE'].apply(lambda x: map_eoh_a_ine.get(x, x))

#Revisamos el largo (nº de caracteres) diferentes, si esta correcto deberian ser solo 5
INE_localidades['COD_INE'].str.len().value_counts()


COD_INE
5    104
Name: count, dtype: int64

Con los codigos corregidos, realizamos el merge en los datos de nivel municipio

In [588]:
def añadir_prefijo_col(df, id, prefijo):
    """Fórmula para incluir prefijos en los df antes de realizar el merge()
    df = dataframe
    id= columna de identificador (COD_INE O COD_PROV)
    prefijo: prefijo a incluir"""
    
    df[id] = df[id].astype('Int64')
    return df.set_index(id).add_prefix(prefijo).reset_index()

In [589]:
# COD_INE, en municipios_data debe ser un int64 para cruzarlo con municipiosDF
municipios_data['COD_INE'] = municipios_data['COD_INE'].astype('Int64')

#formateamos el valor de COD_INE para cruzarlo correctamente
municipiosDF['COD_INE'] = municipiosDF['COD_INE'].astype(str).str[:-6].astype('Int64')

municipiosDF = municipiosDF.merge(añadir_prefijo_col(municipios_data, 
                                                     'COD_INE', 
                                                     'Personas_'),
                                  left_on='COD_INE',
                                  right_on= 'COD_INE')

municipiosDF = municipiosDF.merge(añadir_prefijo_col(INE_localidades, 
                                                     'COD_INE', 
                                                     'Perc_'),
                                  left_on='COD_INE',
                                  right_on= 'COD_INE')
municipiosDF.head()

,COD_INE,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,...,Personas_Unidad,Perc_LOCALIDAD,Perc_VALOR PERNOCTACIONES EXTRANJERO,Perc_VALOR PERNOCTACIONES NACIONAL,Perc_VALOR VIAJERO EXTRANJERO,Perc_VALOR VIAJERO NACIONAL,Perc_PERIODO PERNOCTACIONES EXTRANJERO,Perc_PERIODO PERNOCTACIONES NACIONAL,Perc_PERIODO VIAJERO EXTRANJERO,Perc_PERIODO VIAJERO NACIONAL
0,1059,1001,1,Araba/Álava,Vitoria-Gasteiz,260402,"27696,48",112028,1059006301,Vitoria-Gasteiz,...,Personas.,Vitoria-Gastéiz,175065.0,293398.0,57422.0,136922.0,M07,M04,M07,M04
1,2003,2001,2,Albacete,Albacete,175068,"112698,96",250011,2003000201,Albacete,...,Personas.,Albacete,32020.0,171794.0,16601.0,111280.0,M02,M05,M04,M05
2,3014,3001,3,Alacant/Alicante,Alacant/Alicante,365586,"20257,55",151872,3014000201,Alacant/Alicante,...,Personas.,Alicante,1063874.0,367537.0,360259.0,181608.0,M07,M06,M05,M06
3,3018,3085,3,Alacant/Alicante,Altea,24731,"3465,36",35786,3018000102,Altea,...,Personas.,Altea,178244.0,149324.0,59581.0,72404.0,M09,M08,M05,M07
4,3031,3150,3,Alacant/Alicante,Benidorm,77211,"3850,9874",39091,3031000102,Benidorm,...,Personas.,Benidorm,4105484.0,2238270.0,834735.0,551668.0,M07,M07,M05,M07


In [590]:
provinciasDF = provinciasDF.merge(añadir_prefijo_col(f_INE_provincias, 
                                                     'COD_PROV', 
                                                     'Perc_'),
                                  left_on='COD_PROV',
                                  right_on= 'COD_PROV')

provinciasDF = provinciasDF.merge(añadir_prefijo_col(rentabilidad_H, 
                                                     'COD_INE', 
                                                     'Ratios_'),
                                  left_on='COD_PROV',
                                  right_on= 'COD_INE')

provinciasDF.head()

,COD_PROV,PROVINCIA,COD_CA,COMUNIDAD_AUTONOMA,CAPITAL,Perc_Provincia,Perc_VALOR PERNOCTACIONES,Perc_VALOR VIAJERO,Perc_PERIODO PERNOCTACIONES,Perc_PERIODO VIAJERO,...,Ratios_Provincia,Ratios_Periodo ADR Dato,Ratios_Periodo ADR Tasa Var,Ratios_Periodo RevPar Dato,Ratios_Periodo RevPar Tasa Var,Ratios_Valor ADR Dato,Ratios_Valor ADR Tasa Var,Ratios_Valor RevPar Dato,Ratios_Valor RevPar Tasa Var,Ratios_ADR Anual Medio
0,1,Araba/Álava,16,País Vasco/Euskadi,Vitoria-Gasteiz,Alava,982555.0,456280.0,M08,M08,...,ArabaÁlava,M07,M06,M07,M10,117.12,13.91,89.31,27.88,92.070833
1,2,Albacete,8,Castilla-La Mancha,Albacete,Albacete,802323.0,373917.0,M09,M09,...,Albacete,M09,M11,M09,M09,75.62,20.13,50.10,54.79,66.487500
2,3,Alacant/Alicante,10,Comunitat Valenciana,Alacant/Alicante,Alicante,16590966.0,4345693.0,M08,M08,...,AlicanteAlacant,M08,M11,M08,M11,159.22,8.13,136.07,11.54,106.207500
3,4,Almería,1,Andalucía,Almería,Almería,4567704.0,1372225.0,M08,M08,...,Almería,M08,M09,M08,M09,163.14,16.26,137.15,19.80,89.287500
4,5,Ávila,7,Castilla y León,Ávila,Avila,563816.0,355110.0,M08,M08,...,Ávila,M04,M06,M08,M03,75.19,6.75,36.13,30.83,69.874167


Para finalizar esta parte del proyecto, guardamos los archivos ya trabajados en la carpeta Datos Procesados

In [591]:
# Finalmente, guardamos todos los DataFrames ya revisados y procesados

carpeta = Path('Datos procesados')
carpeta.mkdir(parents=True, exist_ok=True) # Por si no existe, se crea la carpeta



# Datos geográficos
municipiosDF.to_csv(carpeta / 'MunicipiosGeo.csv', index=False, sep=';', encoding='UTF-8') #ok
provinciasDF.to_csv(carpeta / 'ProvinciasGeo.csv', index=False, sep=';', encoding='UTF-8')


#municipios_data.to_csv(carpeta / 'MunicipiosData.csv', index=False, sep=';', encoding='UTF-8')
#INE_provincias.to_csv(carpeta / 'INE_Provincias.csv', index=False, sep=';', encoding='UTF-8')
#INE_localidades.to_csv(carpeta / 'INE_Municipios.csv', index=False, sep=';', encoding='UTF-8')
#f_INE_provincias.to_csv(carpeta / 'INE_Provincias_Pernoctaciones.csv', index=False, sep=';', encoding='UTF-8')
#rentabilidad_H.to_csv(carpeta / 'Provincias_Rentabilidad_Hotelera.csv', index=False, sep=';', encoding='UTF-8')
